In [ ]:
%pip install -r requirements.txt

In [ ]:
# ============================================
# LOCAL LLM AGENT - COMPLETE WORKING CODE
# ============================================

%pip install langchain-community
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
import tiktoken
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json

# ============================================
# CONFIGURATION
# ============================================

# Choose your backend
USE_OLLAMA = True  # Set to True for local Ollama, False for OpenRouter

# Ollama Configuration (for local LLM)
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_MODEL = "qwen3:1.7b"  # Options: "llama3.2:latest", "qwen2.5:7b", "mistral:latest"

# OpenRouter Configuration (optional cloud backup)
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = "your-openrouter-api-key-here"  # Get from https://openrouter.ai/keys
MODEL_PRIMARY = "meta-llama/llama-3.2-3b-instruct:free"

# ============================================
# LLM INITIALIZATION
# ============================================

def get_llm(temp=0.3):
    """Initialize LLM based on configuration"""
    if USE_OLLAMA:
        print(f"🦙 Using local Ollama with model: {OLLAMA_MODEL}")
        return ChatOpenAI(
            model=OLLAMA_MODEL,
            base_url=OLLAMA_BASE_URL,
            api_key='ollama',  # Placeholder, Ollama doesn't need real key
            temperature=temp
        )
    else:
        print(f"☁️ Using OpenRouter with model: {MODEL_PRIMARY}")
        return ChatOpenAI(
            model=MODEL_PRIMARY,
            base_url=OPENROUTER_BASE_URL,
            api_key=OPENROUTER_API_KEY,
            temperature=temp,
            default_headers={'HTTP-Referer': 'http://localhost:8000'}  # Your app URL
        )

# ============================================
# TOOLS FOR THE AGENT
# ============================================

@tool
def calculate(expression: str) -> str:
    """
    Calculate a mathematical expression.
    
    Args:
        expression: Mathematical expression like "25 * 4" or "sqrt(16) + 10"
    
    Returns:
        Calculated result or error message
    """
    import math
    
    # Safe math functions
    safe_dict = {
        'abs': abs, 'round': round, 'min': min, 'max': max,
        'sum': sum, 'pow': pow, 'int': int, 'float': float,
        'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos,
        'tan': math.tan, 'log': math.log, 'exp': math.exp,
        'pi': math.pi, 'e': math.e
    }
    
    try:
        # Evaluate safely
        result = eval(expression, {"__builtins__": {}}, safe_dict)
        return f"✅ Result: {result}"
    except Exception as e:
        return f"❌ Error: {str(e)}"

@tool
def get_current_time() -> str:
    """Get the current date and time"""
    now = datetime.now()
    return f"📅 Current time: {now.strftime('%Y-%m-%d %H:%M:%S')}"

@tool
def get_current_date() -> str:
    """Get the current date only"""
    now = datetime.now()
    return f"📆 Today's date: {now.strftime('%B %d, %Y')}"

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """
    Convert currency using fixed rates (for demo purposes).
    In production, you'd use an API.
    
    Args:
        amount: Amount to convert
        from_currency: Source currency (USD, EUR, GBP, AED)
        to_currency: Target currency
    
    Returns:
        Converted amount
    """
    # Fixed exchange rates (simplified for demo)
    rates = {
        'USD': 1.0, 'EUR': 0.92, 'GBP': 0.79, 'AED': 3.67,
        'INR': 83.5, 'JPY': 150.2, 'CAD': 1.35
    }
    
    from_currency = from_currency.upper()
    to_currency = to_currency.upper()
    
    if from_currency not in rates or to_currency not in rates:
        return f"❌ Unsupported currency. Supported: {list(rates.keys())}"
    
    converted = amount * (rates[to_currency] / rates[from_currency])
    return f"💰 {amount} {from_currency} = {converted:.2f} {to_currency}"

@tool
def analyze_numbers(numbers: str) -> str:
    """
    Analyze a list of numbers.
    
    Args:
        numbers: Comma-separated numbers like "10,20,30,40,50"
    
    Returns:
        Statistical analysis of the numbers
    """
    try:
        num_list = [float(x.strip()) for x in numbers.split(',')]
        
        if not num_list:
            return "❌ No numbers provided"
        
        analysis = {
            'count': len(num_list),
            'sum': sum(num_list),
            'average': sum(num_list) / len(num_list),
            'min': min(num_list),
            'max': max(num_list),
            'range': max(num_list) - min(num_list)
        }
        
        result = f"📊 Analysis of {len(num_list)} numbers:\n"
        for key, value in analysis.items():
            result += f"  • {key.capitalize()}: {value:.2f}\n"
        
        return result
    except Exception as e:
        return f"❌ Error analyzing numbers: {str(e)}"

@tool
def create_note(title: str, content: str) -> str:
    """
    Create a simple note (simulated).
    
    Args:
        title: Note title
        content: Note content
    
    Returns:
        Confirmation message
    """
    note = {
        'title': title,
        'content': content,
        'timestamp': datetime.now().isoformat(),
        'id': hash(f"{title}{datetime.now()}")
    }
    
    # In real implementation, save to database or file
    print(f"\n📝 Note saved: {title}")
    print(f"   Content: {content[:50]}...")
    
    return f"✅ Note '{title}' created successfully!"

# ============================================
# CREATE THE AGENT
# ============================================

def create_local_agent():
    """Create and configure the local LLM agent"""
    
    # Initialize LLM
    llm = get_llm(temp=0.3)
    
    # List all available tools
    tools = [
        calculate,
        get_current_time,
        get_current_date,
        convert_currency,
        analyze_numbers,
        create_note
    ]
    
    print(f"\n🤖 Creating agent with {len(tools)} tools...")
    for tool in tools:
        print(f"  • {tool.name}: {tool.description[:50]}...")
    
    # Create the agent (modern LangChain v1.0+ approach)
    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt="""You are a helpful AI assistant with access to various tools.
        
When responding to users:
1. Use tools when appropriate to provide accurate information
2. Show your reasoning step by step
3. If you need to calculate something, use the calculate tool
4. For currency conversions, ask for amount and currencies if not specified
5. Provide clear, friendly responses
6. If you're unsure, ask clarifying questions

You have access to calculation, time/date, currency conversion, number analysis, and note-taking tools.
"""
    )
    
    return agent

# ============================================
# INTERACTION FUNCTIONS
# ============================================

def chat_with_agent(agent, user_input):
    """Send a message to the agent and get response"""
    try:
        response = agent.invoke({
            "messages": [{"role": "user", "content": user_input}]
        })
        
        # Extract the assistant's response
        if "messages" in response:
            for msg in response["messages"]:
                if msg.get("role") == "assistant":
                    return msg.get("content", "No response generated")
        
        return str(response)
    except Exception as e:
        return f"❌ Error: {str(e)}"

def interactive_session():
    """Run an interactive chat session with the agent"""
    print("\n" + "="*60)
    print("🤖 LOCAL LLM AGENT - INTERACTIVE SESSION")
    print("="*60)
    print("\nCommands:")
    print("  • Type your question normally")
    print("  • Type 'exit' to quit")
    print("  • Type 'tools' to see available tools")
    print("  • Type 'clear' to clear conversation")
    print("-"*60)
    
    agent = create_local_agent()
    conversation_history = []
    
    while True:
        user_input = input("\n💬 You: ").strip()
        
        if user_input.lower() == 'exit':
            print("\n👋 Goodbye!")
            break
        
        elif user_input.lower() == 'tools':
            print("\n🔧 Available Tools:")
            tools = [
                "calculate(expression) - Math calculations",
                "get_current_time() - Get current time",
                "get_current_date() - Get current date",
                "convert_currency(amount, from, to) - Currency conversion",
                "analyze_numbers(numbers) - Statistical analysis",
                "create_note(title, content) - Save a note"
            ]
            for tool in tools:
                print(f"  • {tool}")
            continue
        
        elif user_input.lower() == 'clear':
            conversation_history = []
            print("🧹 Conversation cleared!")
            continue
        
        # Get response from agent
        print("\n🤖 Agent: ", end="", flush=True)
        response = chat_with_agent(agent, user_input)
        print(response)
        
        # Store in history
        conversation_history.append({"user": user_input, "assistant": response})

# ============================================
# DEMO FUNCTIONS
# ============================================

def demo_basic_queries():
    """Demonstrate basic agent capabilities"""
    print("\n" + "="*60)
    print("🎯 DEMO: BASIC AGENT QUERIES")
    print("="*60)
    
    agent = create_local_agent()
    
    test_queries = [
        "What's 25 * 4?",
        "What is 144 divided by 12?",
        "What time is it right now?",
        "What's today's date?",
        "Analyze these numbers: 10, 20, 30, 40, 50",
        "Convert 100 USD to AED"
    ]
    
    for query in test_queries:
        print(f"\n📝 Query: {query}")
        print("-" * 40)
        response = chat_with_agent(agent, query)
        print(f"🤖 Response: {response}")

def demo_complex_tasks():
    """Demonstrate multi-step agent tasks"""
    print("\n" + "="*60)
    print("🧩 DEMO: COMPLEX MULTI-STEP TASKS")
    print("="*60)
    
    agent = create_local_agent()
    
    complex_queries = [
        "Calculate 15 * 8, then tell me the current time",
        "Take these numbers 5, 12, 8, 20, 15, find their average, then convert 50 USD to EUR",
        "What's 100 + 200? Then create a note titled 'Math Result' with that answer",
        "Analyze these sales numbers: 1000, 1500, 2000, 1750, and tell me the total"
    ]
    
    for query in complex_queries:
        print(f"\n🎯 Task: {query}")
        print("-" * 40)
        response = chat_with_agent(agent, query)
        print(f"✅ Result: {response}")

def compare_llm_vs_agent():
    """Compare LLM pipeline vs Agent approach"""
    print("\n" + "="*60)
    print("📊 LLM PIPELINE vs AGENTIC AI — COMPARISON")
    print("="*60)
    
    tasks = [
        ('Summarize a paragraph', 'LLM Pipeline', 'Single call, predictable, fast'),
        ('Answer from documents', 'LLM Pipeline', 'RAG chain — still one direction'),
        ('Research + calculate + report', 'Agent', 'Multi-step, tools needed'),
        ('Monitor + alert on anomalies', 'Agent', 'Continuous loop + decisions'),
        ('Classify support tickets', 'LLM Pipeline', 'No branching needed'),
        ('Resolve support tickets', 'Agent', 'May need to look up account, calculate'),
    ]
    
    print("\nTask Decision Matrix:")
    for task, arch, reason in tasks:
        marker = '🤖' if arch == 'Agent' else '⛓️'
        print(f"  {marker} [{arch:<15}] {task:<35} {reason}")
    
    print("\n💡 Key Insight:")
    print("  • Use LLM Pipeline for deterministic, single-step tasks")
    print("  • Use Agent for tasks requiring tool use, reasoning, or multi-step planning")
    print("  • Agent overhead is higher but enables complex workflows")

# ============================================
# ADVANCED: AGENT WITH MEMORY
# ============================================

class AgentWithMemory:
    """Wrapper for agent with persistent memory"""
    
    def __init__(self):
        self.llm = get_llm(temp=0.3)
        self.tools = [
            calculate, get_current_time, get_current_date,
            convert_currency, analyze_numbers, create_note
        ]
        self.agent = create_agent(
            model=self.llm,
            tools=self.tools,
            system_prompt="You are a helpful assistant with memory of past conversations."
        )
        self.conversation_history = []
    
    def chat(self, user_input):
        """Chat with memory"""
        # Add user input to history
        self.conversation_history.append({"role": "user", "content": user_input})
        
        # Create context from recent history
        context = "\n".join([
            f"{msg['role']}: {msg['content']}" 
            for msg in self.conversation_history[-5:]  # Last 5 messages
        ])
        
        # Get response
        response = chat_with_agent(self.agent, user_input)
        
        # Add to history
        self.conversation_history.append({"role": "assistant", "content": response})
        
        return response

# ============================================
# MAIN EXECUTION
# ============================================

if __name__ == "__main__":
    print("\n" + "="*60)
    print("🚀 LOCAL LLM AGENT FRAMEWORK")
    print("="*60)
    
    # Check if Ollama is running
    if USE_OLLAMA:
        print("\n⚠️  Make sure Ollama is running!")
        print("   Run these commands if not:")
        print("   1. ollama serve")
        print("   2. ollama pull llama3.2")
        print("-"*60)
    
    # Menu
    while True:
        print("\n📋 MAIN MENU")
        print("1. Interactive Chat Session")
        print("2. Run Basic Demos")
        print("3. Run Complex Task Demos")
        print("4. Compare LLM vs Agent")
        print("5. Advanced Agent with Memory")
        print("6. Exit")
        
        choice = input("\n👉 Select option (1-6): ").strip()
        
        if choice == '1':
            interactive_session()
        elif choice == '2':
            demo_basic_queries()
        elif choice == '3':
            demo_complex_tasks()
        elif choice == '4':
            compare_llm_vs_agent()
        elif choice == '5':
            print("\n🧠 Starting Agent with Memory...")
            memory_agent = AgentWithMemory()
            print("Agent ready! Type 'exit' to return to menu.")
            while True:
                user_input = input("\n💬 You: ").strip()
                if user_input.lower() == 'exit':
                    break
                response = memory_agent.chat(user_input)
                print(f"🤖 Agent: {response}")
        elif choice == '6':
            print("\n👋 Thank you for using Local LLM Agent Framework!")
            break
        else:
            print("❌ Invalid choice. Please try again.")

Note: you may need to restart the kernel to use updated packages.


C:\Users\AL chat\AppData\Local\Temp\ipykernel_20124\1586394009.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory



🚀 LOCAL LLM AGENT FRAMEWORK

⚠️  Make sure Ollama is running!
   Run these commands if not:
   1. ollama serve
   2. ollama pull llama3.2
------------------------------------------------------------

📋 MAIN MENU
1. Interactive Chat Session
2. Run Basic Demos
3. Run Complex Task Demos
4. Compare LLM vs Agent
5. Advanced Agent with Memory
6. Exit

🤖 LOCAL LLM AGENT - INTERACTIVE SESSION

Commands:
  • Type your question normally
  • Type 'exit' to quit
  • Type 'tools' to see available tools
  • Type 'clear' to clear conversation
------------------------------------------------------------
🦙 Using local Ollama with model: qwen3:1.7b

🤖 Creating agent with 6 tools...
  • calculate: Calculate a mathematical expression.

Args:
    ex...
  • get_current_time: Get the current date and time...
  • get_current_date: Get the current date only...
  • convert_currency: Convert currency using fixed rates (for demo purpo...
  • analyze_numbers: Analyze a list of numbers.

Args:
    numbers: Com...


pip install langchain langchain-community langchain-openai

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)


In [ ]:
from langchain_core.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def get_weather(city: str) -> str:
    """Get weather for a city"""
    return f"Sunny in {city}"


Step 3 — Create Qwen LLM Wrapper (Custom)
LangChain doesn’t natively support Ollama tools → we wrap it 👇

In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage

import json

class QwenOllama(BaseChatModel):

    def _generate(self, messages, stop=None):
        response = client.chat.completions.create(
            model="qwen3:1.7b",
            messages=[
                {"role": m.type, "content": m.content}
                for m in messages
            ],
            tools=[
                {
                    "type": "function",
                    "function": {
                        "name": "add_numbers",
                        "description": "Add two numbers",
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "a": {"type": "integer"},
                                "b": {"type": "integer"}
                            },
                            "required": ["a", "b"]
                        }
                    }
                },
                {
                    "type": "function",
                    "function": {
                        "name": "get_weather",
                        "description": "Get weather",
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "city": {"type": "string"}
                            },
                            "required": ["city"]
                        }
                    }
                }
            ],
            tool_choice="auto"
        )

        msg = response.choices[0].message

        return AIMessage(
            content=msg.content or "",
            additional_kwargs={
                "tool_calls": msg.tool_calls
            }
        )

# 🚀 LangChain + OpenRouter + Agent Fundamentals

## 📌 1. What is LangChain?

**LangChain** is a framework for building applications powered by LLMs.

It provides:

- ✅ LLM integrations (OpenAI, Ollama, OpenRouter, etc.)
- ✅ Tools
- ✅ Agents
- ✅ Memory
- ✅ Chains (pipelines)

---

# 🧠 Key Components of an LLM System (Ollama + Qwen3:1.7B)

Below is a **clean, practical breakdown** of each component using  
✅ **Ollama (local)**  
✅ **Qwen3:1.7B model**  
✅ **LangChain**

---

# 1️⃣ LLM (Brain of the System)

The LLM is responsible for:
- understanding input
- generating responses
- reasoning

### ✅ Code (Ollama + Qwen3:1.7B)

```python
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(
    model="qwen3:1.7b",
    temperature=0
)

response = llm.invoke("Explain what an AI agent is")
print(response.content)
``


In [ ]:
%pip install langchain-ollama


## 🤖 2. What is an Agent?

An **agent** is an LLM system that can:

1. **Think (reason)**
2. **Decide (what to do)**
3. **Act (call tools)**
4. **Observe (results)**
5. **Repeat until done**

In [ ]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent

# LLM
llm = ChatOllama(
    model="qwen3:1.7b",
    temperature=0
)

# Tool
@tool
def add_numbers(text: str) -> str:
    """Add numbers like '5 10 20'"""
    try:
        return str(sum(map(int, text.split())))
    except:
        return "Invalid input"

tools = [add_numbers]

# Agent
agent = create_agent(
    model=llm,
    tools=tools
)

# Run
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Add 10 20 30"}
    ]
})

# ✅ Correct extraction
print(response["messages"][-1].content)

# chain

In [27]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate  # 

# LLM
llm = ChatOllama(model="qwen3:1.7b", temperature=0)

# Prompt Template
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms."
)

# Format + run
final_prompt = prompt.format(topic="Machine Learning")
response = llm.invoke(final_prompt)

print(response.content)

Machine learning is like a smart teacher that learns from data to make predictions or decisions without being told exactly what to do. Imagine you have a bunch of clues (like weather data), and the teacher (a computer) looks for patterns in those clues to guess something else (like tomorrow's weather). The teacher keeps improving its guesses over time by learning from mistakes. 

It's like a detective who solves mysteries by looking at clues (data) and figuring out what happened (predictions) without being told exactly what to do. The detective uses patterns and trends to solve cases, just like a computer uses data to learn and make better guesses. 

In short, machine learning is a way for computers to "learn" from data to help them make smart decisions or predictions, all without being explicitly told what to do.
